# Complex scalar Helmholtz fields

The scalar pressure is $p=(1+i)(1+x+2y)$ with unit density and bulk modulus,
frequency $\omega=1/2$ and source $f=-\omega^2p$. The skeleton uses two
interleaved real components for every complex mode. Positive frequency removes
the constant-pressure gauge of pure diffusion.

This low-frequency analytical patch checks complex pressure and gradient
coordinates. It does not certify wave resolution, resonance avoidance or an
absorbing boundary. The acoustic applications and convergence records are in
the numbered Helmholtz notebooks beside this example.


In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "src/pymhm").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Declare and solve complex Helmholtz equations

The primary native forms use the exact real/imaginary embedding of complex
pressure; no complex coefficient is discarded. Unit density/modulus give
$-\Delta p-\omega^2p=f$ with $\omega=1/2$ and the independent affine
source $f=-\omega^2p$. This chosen finite patch has invertible local operators;
positive frequency alone does not exclude local resonances in other inputs.

$$
\begin{aligned}
a_T(p,v)&=(\nabla p,\nabla v)_T-\omega^2(p,v)_T,\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The signed trace is $-\nabla p\cdot n$ in two interleaved real components.
The global RHS explicitly prescribes negative complex Dirichlet moments.
The native example has no absorption, PML or resonance-avoidance claim.


In [ ]:
import importlib.util
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows, compile_form
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.fem.scalar.operators import boundary_data
from examples.variational_darcy import NativeFormProvider

native_available = importlib.util.find_spec("dolfinx") is not None
if native_available:
    import ufl
    from dolfinx import fem

    mesh = TriangleMesh.unit_square()
    trace_space = SkeletonSpace(mesh, components=2)
    omega = 0.5

    def exact_components(points):
        """Return real/imaginary components of (1+i)(1+x+2y)."""
        scalar = 1 + points[:, 0] + 2 * points[:, 1]
        return np.column_stack((scalar, scalar))

    def local_forms(context):
        """Write both real components of the complex Helmholtz equation."""
        space = fem.functionspace(context.space.mesh, ("Lagrange", 2, (2,)))
        u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
        x = ufl.SpatialCoordinate(context.space.mesh)
        dx = ufl.dx(domain=context.space.mesh)
        exact = ufl.as_vector((1 + x[0] + 2 * x[1], 1 + x[0] + 2 * x[1]))
        return LocalEquations(
            (ufl.inner(ufl.grad(u), ufl.grad(v)) - omega**2 * ufl.inner(u, v)) * dx,
            ufl.inner(-(omega**2) * exact, v) * dx,
            columns(
                *(
                    float(sign) * v[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for j in range(2)
                )
            ),
            rows(
                *(
                    -float(sign) * u[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for j in range(2)
                )
            ),
            context.trace_dofs,
            metadata={"points": space.tabulate_dof_coordinates()[:, :2].copy()},
        )

    boundary, _ = boundary_data(trace_space, exact_components)
    problem = MultiscaleProblem(
        Equation(0, -boundary),
        NativeFormProvider(mesh, trace_space, local_forms, 2),
        range(len(mesh.cells)),
        trace_space.size,
        (0,) * len(mesh.cells),
    )
    system = assemble(problem)
    solution = system.solve()
    errors = [
        float(np.max(np.abs(field.reshape(-1, 2) - exact_components(record["points"]))))
        for field, record in zip(solution.fields, system.local_metadata, strict=True)
    ]
    assert max(errors) < 1e-10
    print(
        {
            "native_available": True,
            "spaces": "complex P2 pressure / P0 complex trace",
            "maximum_component_nodal_error_by_cell": errors,
            "global_original_relative_residual": solution.raw_residual,
        }
    )
else:
    print(
        {
            "native_available": False,
            "status": "User-written UFL primary solve not executed: select the Pixi fem kernel.",
        }
    )


## Compare the supported methods
These comparisons invoke the existing formulation implementations through the application catalogue and preserve their numerical records. The user-defined primary UFL solve above exercises the generic local/global API; the catalogue drivers remain compatibility comparisons.


In [ ]:
from examples.tutorial_scalar_variants import (
    VARIANTS,
    affine_pressure,
    affine_flux,
    run_variant,
)

record = run_variant("helmholtz")
print(record["spaces_and_conventions"])
print(record["metrics"])
